# 第 05 章：为运行事实划清所有权（Context Engineering）（概念实验与工程迁移）

按正文顺序完成每个实验：先写预测，再运行代码，阅读输出，最后修改一个变量。

概念实验不会预先导入 Mini DeerFlow；进入“工程迁移”标签后，才把同一机制放回项目。

## 实验 1：把身份、Token 和数据库连接一起塞进 State

`concept` · `failure` · `context-boundaries`

**运行前先预测**：如果 State 被 checkpoint serializer 处理，Token 会不会仍然可见？`sqlite3.Connection` 能否被序列化？

> 先在这里写下你的判断，再执行下一个代码单元。

In [1]:
import sqlite3

from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


unsafe_connection = sqlite3.connect(":memory:")
universal_state = {
    "user_id": "learner-1",
    "auth_token": "chapter05-secret",
    "connection": unsafe_connection,
    "messages": ["研究 checkpoint"],
    "plan": ["检索", "汇总"],
    "language": "zh-CN",
}

print("checkpoint_fields =", sorted(universal_state))
print("secret_visible_in_state =", "auth_token" in universal_state)
try:
    JsonPlusSerializer().dumps_typed(universal_state)
except TypeError as error:
    assert "sqlite3.Connection" in str(error)
    print("TypeError: sqlite3.Connection is not checkpoint serializable")
else:
    raise AssertionError("数据库连接不应进入 checkpoint State")
finally:
    unsafe_connection.close()


checkpoint_fields = ['auth_token', 'connection', 'language', 'messages', 'plan', 'user_id']
secret_visible_in_state = True
TypeError: sqlite3.Connection is not checkpoint serializable


**发生了什么**：同一个设计同时制造了两个问题。Token 成为可持久化 State 的普通字段；数据库连接则无法通过 LangGraph 的 checkpoint serializer。
“加密 checkpoint”不能解决全部问题。模型、节点、trace 和调试工具仍可能读到本不该出现的值；连接对象也不是需要恢复的业务事实。

**动手修改**：先只删除 `connection`，再预测 Token 是否已经安全。列出仍能读取 `auth_token` 的组件。

## 实验 2：用原生 Runtime 拆开运行依赖与线程事实

`concept` · `repair` · `context-boundaries`

**运行前先预测**：节点返回 patch 后，最终 State 会不会出现 `auth_token` 或数据库连接？

> 先在这里写下你的判断，再执行下一个代码单元。

In [2]:
from dataclasses import dataclass, field
import sqlite3
from typing import TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime


@dataclass(frozen=True)
class ResearchContext:
    user_id: str
    permissions: frozenset[str]
    connection: sqlite3.Connection = field(repr=False)
    auth_token: str = field(repr=False)


class ResearchState(TypedDict):
    objective: str
    plan: list[str]


def build_plan(
    state: ResearchState,
    runtime: Runtime[ResearchContext],
) -> dict[str, list[str]]:
    context = runtime.context
    assert context.connection.execute("SELECT 1").fetchone() == (1,)
    print("context_user =", context.user_id)
    print("context_permissions =", sorted(context.permissions))
    patch = {"plan": [f"为 {state['objective']} 检索资料", "整理引用"]}
    print("state_patch =", patch)
    return patch


context_builder = StateGraph(ResearchState, context_schema=ResearchContext)
context_builder.add_node("build_plan", build_plan)
context_builder.add_edge(START, "build_plan")
context_builder.add_edge("build_plan", END)
context_graph = context_builder.compile()

safe_connection = sqlite3.connect(":memory:")
context_result = context_graph.invoke(
    {"objective": "解释 checkpoint", "plan": []},
    context=ResearchContext(
        user_id="learner-1",
        permissions=frozenset({"knowledge:read"}),
        connection=safe_connection,
        auth_token="runtime-only",
    ),
)
safe_connection.close()
print("final_state =", context_result)
print("secret_in_state =", "auth_token" in context_result)


context_user = learner-1
context_permissions = ['knowledge:read']
state_patch = {'plan': ['为 解释 checkpoint 检索资料', '整理引用']}
final_state = {'objective': '解释 checkpoint', 'plan': ['为 解释 checkpoint 检索资料', '整理引用']}
secret_in_state = False


**发生了什么**：应用通过 `context=` 注入身份、权限和连接；节点只能读取 frozen context。节点返回的 `plan` patch 才进入 Graph State。
Context 不会自动进入 Prompt。应用仍要选择哪些安全字段可以给模型；`repr=False` 也只是降低误打印概率，不替代权限检查和日志脱敏。

**动手修改**：尝试在节点中改写 `runtime.context.user_id`，观察 frozen dataclass 如何阻止修改。再解释为什么真正鉴权仍必须发生在服务端。

## 实验 3：把用户偏好放进 Thread State 后切换会话

`concept` · `failure` · `store`

**运行前先预测**：同一个 `user_id` 使用新的 `thread_id` 时，新 Thread 能否读到旧 Thread 的 `language`？

> 先在这里写下你的判断，再执行下一个代码单元。

In [3]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class PreferenceInState(TypedDict, total=False):
    user_id: str
    language: str
    observed_language: str


def read_state_preference(state: PreferenceInState) -> dict[str, str]:
    return {"observed_language": state.get("language", "<missing>")}


preference_state_builder = StateGraph(PreferenceInState)
preference_state_builder.add_node("read_preference", read_state_preference)
preference_state_builder.add_edge(START, "read_preference")
preference_state_builder.add_edge("read_preference", END)
preference_state_graph = preference_state_builder.compile(
    checkpointer=InMemorySaver()
)

pref_thread_a = {"configurable": {"thread_id": "preference-a"}}
pref_thread_b = {"configurable": {"thread_id": "preference-b"}}
pref_a = preference_state_graph.invoke(
    {"user_id": "learner-1", "language": "zh-CN"}, config=pref_thread_a
)
pref_b = preference_state_graph.invoke(
    {"user_id": "learner-1"}, config=pref_thread_b
)
print("thread_a_language =", pref_a["observed_language"])
print("thread_b_language =", pref_b["observed_language"])


thread_a_language = zh-CN
thread_b_language = <missing>


**发生了什么**：Checkpointer 按 `thread_id` 保存 Graph State。相同 `user_id` 只是普通字段，不会让两个 Thread 自动共享 State。
这个结果正好证明了 Thread 隔离。偏好若要跨会话复用，就需要另一个不绑定单一 Thread、由应用显式读写的边界。

**动手修改**：把两个 config 改成相同 `thread_id`。预测结果后说明：这为什么不能作为跨 Thread 偏好的修复方案？

## 实验 4：用同一用户 namespace 跨 Thread 读取偏好

`concept` · `repair` · `store`

**运行前先预测**：Thread A 保存偏好后，Thread B 使用相同用户 Context，能否从 Store 读取？

> 先在这里写下你的判断，再执行下一个代码单元。

In [4]:
from dataclasses import dataclass
from typing import TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore


@dataclass(frozen=True)
class PreferenceContext:
    user_id: str


class PreferenceActionState(TypedDict):
    action: str
    language: str
    observed_language: str


def manage_preference(
    state: PreferenceActionState,
    runtime: Runtime[PreferenceContext],
) -> dict[str, str]:
    namespace = ("users", runtime.context.user_id)
    if state["action"] == "save":
        runtime.store.put(namespace, "preferences", {"language": state["language"]})
    saved = runtime.store.get(namespace, "preferences")
    language = saved.value["language"] if saved else "<missing>"
    return {"observed_language": language}


preference_store = InMemoryStore()
preference_builder = StateGraph(
    PreferenceActionState,
    context_schema=PreferenceContext,
)
preference_builder.add_node("manage_preference", manage_preference)
preference_builder.add_edge(START, "manage_preference")
preference_builder.add_edge("manage_preference", END)
preference_graph = preference_builder.compile(store=preference_store)

saved_pref = preference_graph.invoke(
    {"action": "save", "language": "zh-CN", "observed_language": ""},
    context=PreferenceContext(user_id="learner-1"),
)
loaded_pref = preference_graph.invoke(
    {"action": "load", "language": "", "observed_language": ""},
    context=PreferenceContext(user_id="learner-1"),
)
print("thread_a_saved =", saved_pref["observed_language"])
print("thread_b_loaded =", loaded_pref["observed_language"])
print("namespace =", ("users", "learner-1"))


thread_a_saved = zh-CN
thread_b_loaded = zh-CN
namespace = ('users', 'learner-1')


**发生了什么**：Store 的 namespace 使用应用提供的用户身份，因此数据生命周期独立于 Thread。代码只保存 `language`，没有把消息、计划和 Token 一起复制进去。

**动手修改**：把保存的 value 扩成任意自由文本。列出长度、字段白名单、删除、Prompt injection 与隐私保留期方面的新风险。

## 实验 5：验证不同用户不会共享偏好

`concept` · `contrast` · `store`

**运行前先预测**：两个 namespace 使用相同 key `preferences`，它们会覆盖还是隔离？

> 先在这里写下你的判断，再执行下一个代码单元。

In [5]:
from langgraph.store.memory import InMemoryStore


isolation_store = InMemoryStore()
isolation_store.put(
    ("users", "learner-1"),
    "preferences",
    {"language": "zh-CN"},
)
isolation_store.put(
    ("users", "learner-2"),
    "preferences",
    {"language": "en-US"},
)

learner_1 = isolation_store.get(("users", "learner-1"), "preferences")
learner_2 = isolation_store.get(("users", "learner-2"), "preferences")
unknown = isolation_store.get(("users", "unknown"), "preferences")
print("learner-1 =", learner_1.value)
print("learner-2 =", learner_2.value)
print("unknown =", unknown)


learner-1 = {'language': 'zh-CN'}
learner-2 = {'language': 'en-US'}
unknown = None


**发生了什么**：key 相同并不会跨 namespace 冲突。隔离是否可靠取决于 namespace 身份是否可信，以及底层 Store 是否执行相应访问控制。

**动手修改**：故意用请求参数中的 `user_id` 替代认证 Context。描述攻击者如何读取另一个用户的 namespace，以及 Gateway 应在哪里拒绝。

## 实验 6：把账户余额复制进 Store 后观察陈旧值

`concept` · `failure` · `business-database`

**运行前先预测**：业务数据库把余额从 100 更新为 60 后，Store 中旧快照会不会自动变化？

> 先在这里写下你的判断，再执行下一个代码单元。

In [6]:
import sqlite3

from langgraph.store.memory import InMemoryStore


account_db = sqlite3.connect(":memory:")
account_db.execute("CREATE TABLE accounts (user_id TEXT PRIMARY KEY, balance INTEGER)")
account_db.execute("INSERT INTO accounts VALUES ('learner-1', 100)")

business_copy_store = InMemoryStore()
business_copy_store.put(
    ("users", "learner-1"),
    "account",
    {"balance": 100},
)

account_db.execute(
    "UPDATE accounts SET balance = 60 WHERE user_id = 'learner-1'"
)
database_balance = account_db.execute(
    "SELECT balance FROM accounts WHERE user_id = 'learner-1'"
).fetchone()[0]
store_balance = business_copy_store.get(
    ("users", "learner-1"), "account"
).value["balance"]

print("database_balance =", database_balance)
print("store_balance =", store_balance)
print("facts_disagree =", database_balance != store_balance)
account_db.close()


database_balance = 60
store_balance = 100
facts_disagree = True


**发生了什么**：Store 正常保存了应用写入的值；错误在于应用把权威业务事实复制成了无人维护的长期记忆。Prompt 可能据此给出错误承诺。

**动手修改**：尝试在每次余额变化时同步更新 Store。列出并发、失败重试、事务和补偿会让这条“双写”方案增加哪些成本。

## 实验 7：每次通过业务 Repository 读取权威余额

`concept` · `repair` · `business-database`

**运行前先预测**：数据库更新后再次调用 `get_balance`，是否还需要同步 Store？

> 先在这里写下你的判断，再执行下一个代码单元。

In [7]:
import sqlite3

from langgraph.store.memory import InMemoryStore


class AccountRepository:
    def __init__(self, connection: sqlite3.Connection) -> None:
        self.connection = connection

    def get_balance(self, user_id: str) -> int:
        row = self.connection.execute(
            "SELECT balance FROM accounts WHERE user_id = ?",
            (user_id,),
        ).fetchone()
        if row is None:
            raise KeyError(user_id)
        return int(row[0])


authority_db = sqlite3.connect(":memory:")
authority_db.execute("CREATE TABLE accounts (user_id TEXT PRIMARY KEY, balance INTEGER)")
authority_db.execute("INSERT INTO accounts VALUES ('learner-1', 100)")
accounts = AccountRepository(authority_db)

preference_only_store = InMemoryStore()
preference_only_store.put(
    ("users", "learner-1"),
    "preferences",
    {"language": "zh-CN"},
)
print("balance_before =", accounts.get_balance("learner-1"))
authority_db.execute(
    "UPDATE accounts SET balance = 60 WHERE user_id = 'learner-1'"
)
print("balance_after =", accounts.get_balance("learner-1"))
print(
    "stored_preference =",
    preference_only_store.get(("users", "learner-1"), "preferences").value,
)
authority_db.close()


balance_before = 100
balance_after = 60
stored_preference = {'language': 'zh-CN'}


**发生了什么**：Repository 是业务数据库的受控访问边界，读取结果随权威事务变化。Store 继续保存语言偏好，两类数据不再争夺“真相来源”。

**动手修改**：让 Repository 返回退款状态，并要求工具执行退款。指出读取、权限检查、幂等键、事务和审计分别应由哪一层拥有。

## 实验 8：用两个 thread_id 保存互不相同的研究问题

`concept` · `contrast` · `thread-state`

**运行前先预测**：读取 Thread A 的 snapshot 时，会不会出现 Thread B 的问题或答案？

> 先在这里写下你的判断，再执行下一个代码单元。

In [8]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class ThreadResearchState(TypedDict):
    question: str
    answer: str


def answer_question(state: ThreadResearchState) -> dict[str, str]:
    return {"answer": f"已记录：{state['question']}"}


thread_builder = StateGraph(ThreadResearchState)
thread_builder.add_node("answer", answer_question)
thread_builder.add_edge(START, "answer")
thread_builder.add_edge("answer", END)
thread_graph = thread_builder.compile(checkpointer=InMemorySaver())

thread_a_config = {"configurable": {"thread_id": "research-a"}}
thread_b_config = {"configurable": {"thread_id": "research-b"}}
thread_graph.invoke(
    {"question": "解释 checkpoint", "answer": ""}, config=thread_a_config
)
thread_graph.invoke(
    {"question": "解释 store", "answer": ""}, config=thread_b_config
)
snapshot_a = thread_graph.get_state(thread_a_config).values
snapshot_b = thread_graph.get_state(thread_b_config).values
print("thread_a =", snapshot_a)
print("thread_b =", snapshot_b)
print("questions_isolated =", snapshot_a["question"] != snapshot_b["question"])


thread_a = {'question': '解释 checkpoint', 'answer': '已记录：解释 checkpoint'}
thread_b = {'question': '解释 store', 'answer': '已记录：解释 store'}
questions_isolated = True


**发生了什么**：Checkpointer 以 `thread_id` 组织 State 历史。同一用户可以拥有多个 Thread；一个 Thread 也可能经历多次 Run 和恢复。

**动手修改**：故意让两个请求共用同一个 `thread_id`。观察第二次输入怎样继承旧 snapshot，并解释为什么产品 Thread 必须绑定认证用户。

## 实验 9：对照安全 Context、ThreadState 与偏好 Repository

`migration` · `contrast` · `context-boundaries`

**运行前先预测**：安全视图是否包含 Token？同一路径 Artifact 是否保持类型化？不同用户的偏好是否使用不同 namespace？

> 先在这里写下你的判断，再执行下一个代码单元。

In [9]:
from langgraph.store.memory import InMemoryStore

from mini_deerflow.context import RuntimeContext, safe_context_view
from mini_deerflow.schemas import ArtifactRef
from mini_deerflow.state import MiddlewareTraceEvent, assert_checkpoint_safe
from mini_deerflow.store import UserPreferenceRepository, preference_namespace


project_context = RuntimeContext(
    user_id="learner-1",
    workspace_root="/tmp/mini-deerflow",
    request_id="req-05-001",
    permissions=frozenset({"knowledge:read", "workspace:read"}),
    locale="zh-CN",
    auth_token="never-publish-me",
)
safe_view = safe_context_view(project_context)

project_state = {
    "messages": [],
    "artifacts": [
        ArtifactRef(path="reports/context.md", media_type="text/markdown")
    ],
    "middleware_trace": [
        MiddlewareTraceEvent(middleware="lead", hook="before_model")
    ],
}
assert_checkpoint_safe(project_state)

project_store = InMemoryStore()
preferences = UserPreferenceRepository(project_store)
preferences.save(
    "learner-1",
    {"language": "zh-CN", "citation_style": "source-first"},
)

print("safe_context_keys =", sorted(safe_view))
print("auth_token_exposed =", "auth_token" in safe_view)
print("artifact =", project_state["artifacts"][0].model_dump())
print("preference_namespace =", preference_namespace("learner-1"))
print("preferences =", preferences.load("learner-1"))


safe_context_keys = ['locale', 'model_profile', 'permissions', 'request_id', 'user_id']
auth_token_exposed = False
artifact = {'path': 'reports/context.md', 'media_type': 'text/markdown'}
preference_namespace = ('users', 'learner-1')
preferences = {'language': 'zh-CN', 'citation_style': 'source-first'}


**发生了什么**：Mini DeerFlow 增加了类型化 Context、安全投影视图、checkpoint safety guard、Artifact 协议和受约束偏好 Repository。
概念实验省略的工程边界现在有了拥有者：Gateway 认证身份，Middleware 校验权限，State 类型限制可持久化事实，Repository 执行字段白名单与用户隔离。